## Question 1. Select the Tool

You can use the same tool you used when completing the module,
or choose a different one for your homework.

What's the name of the orchestrator you chose? 

- Perfect

In [1]:
#%pip install prefect

## Question 2. Version

What's the version of the orchestrator? 

In [2]:
!prefect version

Version:              3.8.6
API version:          0.8.4
Python version:       3.14.2
Git commit:           33de4078
Built:                Mon, Sep 14, 2026 02:21 PM
OS/Arch:              linux/x86_64
Profile:              ephemeral
Server type:          ephemeral
Pydantic version:     2.13.5
Server:
  Database:           sqlite
  SQLite version:     3.45.1


## Question 3. Creating a pipeline

Let's read the March 2023 Yellow taxi trips data.

How many records did we load? 

- 3,003,766
- 3,203,766
- 3,403,766
- 3,603,766

In [3]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

/workspaces/mlops-zoomcamp-course/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet"
df = pd.read_parquet(url)
print(f"✅ Loaded {len(df):,} records")

✅ Loaded 3,403,766 records


## Question 4. Data preparation

Let's continue with pipeline creation.

We will use the same logic for preparing the data we used previously. 

This is what we used (adjusted for yellow dataset):

```python
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df
```

Let's apply to the data we loaded in question 3. 

What's the size of the result? 

- 2,903,766
- 3,103,766
- 3,316,216 
- 3,503,766


In [5]:
def prepare_data(df):
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df['duration'].dt.total_seconds() / 60
    df = df[(df['duration'] >= 1) & (df['duration'] <= 60)].copy()
    df[['PULocationID', 'DOLocationID']] = df[['PULocationID', 'DOLocationID']].astype(str)
    return df

df_clean = prepare_data(df)
print(f"✅ Cleaned Rows: {len(df_clean):,}")
#Clear df from memory
del df

✅ Cleaned Rows: 3,316,216


## Question 5. Train a model

We will now train a linear regression model using the same code as in homework 1.

* Fit a dict vectorizer.
* Train a linear regression with default parameters.
* Use pick up and drop off locations separately, don't create a combination feature.

Let's now use it in the pipeline. We will need to create another transformation block, and return both the dict vectorizer and the model.

What's the intercept of the model? 

Hint: print the `intercept_` field in the code block

- 21.77
- 24.77
- 27.77
- 31.77

In [7]:
from sklearn.model_selection import train_test_split

df_clean['target'] = df_clean['duration']
categorical = ['PULocationID', 'DOLocationID']
def df_to_dict(df):
    return df[categorical].to_dict(orient='records')
train_df, val_df = train_test_split(df_clean, test_size=0.2, random_state=42)
dv = DictVectorizer()
X_train = dv.fit_transform(df_to_dict(train_df))
X_val = dv.transform(df_to_dict(val_df))
y_train = train_df['target'].values
y_val = val_df['target'].values

mlflow.set_experiment("homework3-nyc-taxi")

with mlflow.start_run():
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="models",
        input_example=X_val[:1],
        signature=mlflow.models.signature.infer_signature(X_val, y_pred)
    )


2026/09/19 11:05:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [9]:
print(f"✅ RMSE: {rmse:.2f}")
print(f"✅ Intercept: {lr.intercept_}")

✅ RMSE: 8.15
✅ Intercept: 24.751647886022617


## Question 6. Register the model 

The model is trained, so let's save it with MLFlow.

Find the logged model, and find MLModel file. What's the size of the model? (`model_size_bytes` field):

* 14,534
* 9,534
* 4,534
* 1,534


In [10]:
import pickle
import pathlib

output_dir = pathlib.Path("artifacts")
output_dir.mkdir(exist_ok=True)
with open(output_dir / "dv.pkl", "wb") as f_out:
    pickle.dump(dv, f_out)
mlflow.log_artifact(str(output_dir / "dv.pkl"))

In [ ]:
import yaml

mlmodel_path = next(pathlib.Path("mlruns").glob("**/artifacts/MLmodel"))
mlmodel = yaml.safe_load(mlmodel_path.read_text())

print(f"✅ model_size_bytes: {mlmodel['model_size_bytes']}")
#* If your answer doesn't match options exactly, select the closest one.

✅ model_size_bytes: 14414
